# JAX vs. Gazebo (with Rosbag Initial Conditions) Comparison

This notebook directly reads a rosbag file to extract both the Gazebo simulation trajectory and its **initial conditions**. It then runs a JAX simulation starting from the same initial state and compares the results against the ground truth trajectory and the Gazebo data.

**Instructions:**
1.  **Configure Paths:** Set the path to your rosbag file and the trajectory (`.npy`) file.
2.  **Set Controller Gains:** Update the controller gains to match your PX4 Gazebo simulation.
3.  **Run All Cells:** Execute all cells to perform the analysis.

In [2]:
# Add these to your first code cell
import os
import sys
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message

# Attempt to import rosbag2_py components
try:
    from rosbag2_py import SequentialReader, StorageOptions, ConverterOptions
except ImportError:
    print("Could not import rosbag2_py. Please ensure your ROS 2 environment is sourced.")

# Add parent directory to path to allow importing utils and dynamics
sys.path.append('..')

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from functools import partial
import os
import sys
from scipy.spatial.transform import Rotation
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message

# Attempt to import rosbag2_py components
try:
    from rosbag2_py import SequentialReader, StorageOptions, ConverterOptions
except ImportError:
    print("Could not import rosbag2_py. Please ensure your ROS 2 environment is sourced.")

# Configure JAX for 64-bit precision
from jax import config
config.update("jax_enable_x64", True)

# Set up plotting style
plt.style.use('seaborn-v0_8-whitegrid')

# Import from custom modules
try:
    from utils import hat, vee, odeint_fixed_step
    from dynamics import prior
except ImportError:
    print("Please ensure utils.py and dynamics.py are in the same directory.")

## Configuration

In [4]:
# --- File Paths ---
ROSBAG_PATH = "bodyrate_bag"
TRAJECTORY_FILE_PATH = "circle_r2.0_t20s_alt1.5_initpsi0deg_pointToCenter_50hz_11col_2laps.npy"

# --- ROS Topics ---
POSE_TOPIC = "/mavros/local_position/pose"

# --- Controller Gains (ensure these match your PX4 setup) ---
k_R = jnp.array([2.0, 2.0, 1.0])
K_mat = jnp.diag(jnp.array([2.0, 2.0, 3.0]))
Lambda_mat = jnp.diag(jnp.array([1.0, 1.0, 1.0]))

## Load Trajectory and Rosbag Data (including Initial Conditions)

In [6]:
# --- Load Ground Truth Trajectory ---
traj_data = np.load(TRAJECTORY_FILE_PATH)
ts_ref = traj_data[:, 0]
r_ref = traj_data[:, 1:4]
dr_ref = traj_data[:, 4:7]
ddr_ref = traj_data[:, 8:11]

# --- ROS Topics ---
CONTROL_LOG_TOPIC = "/mlac_mission_node/control_log"
POSE_TOPIC = "/mavros/local_position/pose"

def get_rosbag_options(path, storage_id='sqlite3'):
    storage_options = StorageOptions(uri=path, storage_id=storage_id)
    converter_options = ConverterOptions(input_serialization_format='cdr', output_serialization_format='cdr')
    return storage_options, converter_options

# --- Find Trajectory Execution Window from Rosbag ---
reader = SequentialReader()
storage_options, converter_options = get_rosbag_options(ROSBAG_PATH)
reader.open(storage_options, converter_options)

topic_types = {meta.name: meta.type for meta in reader.get_all_topics_and_types()}
ControllerLogMsgClass = get_message(topic_types[CONTROL_LOG_TOPIC])

traj_exec_start_ns = -1
traj_exec_end_ns = -1

while reader.has_next():
    (topic, data, t_ns) = reader.read_next()
    if topic == CONTROL_LOG_TOPIC:
        msg = deserialize_message(data, ControllerLogMsgClass)
        # Check for the start of trajectory tracking
        if traj_exec_start_ns == -1 and msg.trajectory_execution_start_ros_time.sec > 0:
            traj_exec_start_ns = msg.trajectory_execution_start_ros_time.sec * 1e9 + msg.trajectory_execution_start_ros_time.nanosec
        # Check for the end of trajectory tracking
        if msg.trajectory_execution_end_ros_time.sec > 0:
            traj_exec_end_ns = msg.trajectory_execution_end_ros_time.sec * 1e9 + msg.trajectory_execution_end_ros_time.nanosec
            break  # Stop after finding the end time

# --- Extract Filtered Pose Data ---
reader.seek(0) # Reset reader to the beginning
PoseStampedMsgClass = get_message(topic_types[POSE_TOPIC])

t_gazebo, q_gazebo, quat_gazebo = [], [], []
initial_pose_msg = None

while reader.has_next():
    (topic, data, t_ns) = reader.read_next()
    
    # Only process pose messages within the tracking window
    if topic == POSE_TOPIC and traj_exec_start_ns != -1 and t_ns >= traj_exec_start_ns:
        if traj_exec_end_ns != -1 and t_ns > traj_exec_end_ns:
            break # Stop if we are past the tracking window

        msg = deserialize_message(data, PoseStampedMsgClass)
        if initial_pose_msg is None:
            initial_pose_msg = msg
        
        # Time relative to the start of tracking
        t_gazebo.append((t_ns - traj_exec_start_ns) / 1e9)
        q_gazebo.append([msg.pose.position.x, msg.pose.position.y, msg.pose.position.z])
        quat_gazebo.append([msg.pose.orientation.x, msg.pose.orientation.y, msg.pose.orientation.z, msg.pose.orientation.w])

t_gazebo = np.array(t_gazebo)
q_gazebo = np.array(q_gazebo)
quat_gazebo = np.array(quat_gazebo)

euler_gazebo = Rotation.from_quat(quat_gazebo).as_euler('xyz', degrees=True)

print(f"Loaded and filtered Gazebo data: {len(t_gazebo)} points during trajectory tracking.")
if initial_pose_msg:
    print(f"Initial Gazebo position for JAX sim: {q_gazebo[0]}")

[INFO] [1759461556.933712865] [rosbag2_storage]: Opened database 'bodyrate_bag/windy_bag_0921_153654_0.db3' for READ_ONLY.


ModuleNotFoundError: No module named 'mlac_msgs'

## JAX Simulation

In [ ]:
def npy_reference_func(t, ts_ref, r_ref, dr_ref, ddr_ref):
    """Returns desired pos, vel, and acc by interpolating the .npy data."""
    r = jnp.array([jnp.interp(t, ts_ref, r_ref[:, i]) for i in range(3)])
    dr = jnp.array([jnp.interp(t, ts_ref, dr_ref[:, i]) for i in range(3)])
    ddr = jnp.array([jnp.interp(t, ts_ref, ddr_ref[:, i]) for i in range(3)])
    return r, dr, ddr

def simulation_ode(z, t, k_R, K_mat, Lambda_mat, reference_func, dt):
    """Calculates the derivatives of the system state for the ODE solver."""
    x, R_flatten, Omega_state = z
    q, dq = x[:3], x[3:]
    R = R_flatten.reshape((3, 3))
    
    r, dr, ddr = reference_func(t)
    
    e, de = q - r, dq - dr
    s = de + Lambda_mat @ e
    v, dv = dr - Lambda_mat @ e, ddr - Lambda_mat @ de
    H, C, g, B = prior(q, dq)
    tau = H @ dv + C @ v + g - K_mat @ s
    u_d = jnp.linalg.solve(B, tau)

    f_d = jnp.linalg.norm(u_d)
    b_3d = u_d / (f_d + 1e-6)
    
    b_1d_desired = dr / (jnp.linalg.norm(dr) + 1e-6)
    b_2d_temp = jnp.cross(b_3d, b_1d_desired)
    b_2d = b_2d_temp / (jnp.linalg.norm(b_2d_temp) + 1e-6)
    b_1d = jnp.cross(b_2d, b_3d)
    
    R_d = jnp.column_stack((b_1d, b_2d, b_3d))

    e_R = 0.5 * vee(R_d.T @ R - R.T @ R_d)
    Omega_cmd = -k_R * e_R

    dR = R @ hat(Omega_cmd)
    dR_flatten = dR.flatten()
    u_applied = f_d * R @ jnp.array([0., 0., 1.])
    ddq = jnp.linalg.solve(H, u_applied - C @ dq - g)
    dx = jnp.concatenate((dq, ddq))
    
    dOmega = (Omega_cmd - Omega_state) / dt
    
    return dx, dR_flatten, dOmega

@partial(jax.jit, static_argnums=(5, 7, 8, 9, 10, 11))
def jax_flatten_wrapper(z_flat, t, k_R, K_mat, Lambda_mat, reference_func, dt, z_unravel_func, ts_ref, r_ref, dr_ref, ddr_ref):
    z_tree = z_unravel_func(z_flat)
    
    ref_func_partial = partial(reference_func, ts_ref=ts_ref, r_ref=r_ref, dr_ref=dr_ref, ddr_ref=ddr_ref)
    
    dz_tree = simulation_ode(z_tree, t, k_R, K_mat, Lambda_mat, ref_func_partial, dt)
    return jnp.concatenate(jax.tree_util.tree_leaves(dz_tree))

## Run JAX Simulation

In [ ]:
# --- Simulation Parameters ---
T_FINAL = ts_ref[-1]
DT = ts_ref[1] - ts_ref[0]

# --- Initial Conditions from Rosbag ---
if initial_pose_msg:
    # Use the first position from the rosbag
    r0_jax = jnp.array([initial_pose_msg.pose.position.x, 
                         initial_pose_msg.pose.position.y, 
                         initial_pose_msg.pose.position.z])
    # Use the first velocity from the reference trajectory
    dr0_jax = jnp.array(dr_ref[0])
    
    x0_jax = jnp.concatenate([r0_jax, dr0_jax])
    
    # Convert initial quaternion to rotation matrix
    q_init = np.array([initial_pose_msg.pose.orientation.x, 
                       initial_pose_msg.pose.orientation.y, 
                       initial_pose_msg.pose.orientation.z, 
                       initial_pose_msg.pose.orientation.w])
    R0_jax = jnp.array(Rotation.from_quat(q_init).as_matrix())
else:
    # Fallback to reference trajectory initial conditions
    print("Warning: No pose data in rosbag, using trajectory file for initial conditions.")
    x0_jax = jnp.concatenate([r_ref[0], dr_ref[0]])
    R0_jax = jnp.eye(3)

R0_flatten_jax = R0_jax.flatten()
Omega0_jax = jnp.zeros(3)

z0_tree = (x0_jax, R0_flatten_jax, Omega0_jax)
z0_flat, z_unravel_func = jax.flatten_util.ravel_pytree(z0_tree)

print("Starting JAX simulation...")

ts_ref_jnp = jnp.array(ts_ref)
r_ref_jnp = jnp.array(r_ref)
dr_ref_jnp = jnp.array(dr_ref)
ddr_ref_jnp = jnp.array(ddr_ref)

ode_for_solver = partial(
    jax_flatten_wrapper,
    k_R=k_R,
    K_mat=K_mat,
    Lambda_mat=Lambda_mat,
    reference_func=npy_reference_func,
    dt=DT,
    z_unravel_func=z_unravel_func,
    ts_ref=ts_ref_jnp,
    r_ref=r_ref_jnp,
    dr_ref=dr_ref_jnp,
    ddr_ref=ddr_ref_jnp
)

z_history_flat, ts_jax = odeint_fixed_step(ode_for_solver, z0_flat, 0.0, T_FINAL, DT)

z_history = jax.vmap(z_unravel_func)(z_history_flat)
x_hist, R_flat_hist, _ = z_history
q_jax = x_hist[:, :3]
R_jax = R_flat_hist.reshape(-1, 3, 3)
euler_jax = Rotation.from_matrix(np.asarray(R_jax)).as_euler('xyz', degrees=True)

print("JAX simulation complete!")

## Comparison Plots

In [ ]:
# Plot 1: 3D Trajectory Comparison
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
ax.plot(r_ref[:, 0], r_ref[:, 1], r_ref[:, 2], 'r--', label='Ground Truth Trajectory')
ax.plot(q_jax[:, 0], q_jax[:, 1], q_jax[:, 2], 'b-', label='JAX Simulated Trajectory', lw=2)
ax.plot(q_gazebo[:, 0], q_gazebo[:, 1], q_gazebo[:, 2], 'g:', label='Gazebo (Rosbag) Trajectory', lw=2)
ax.set_title('3D Trajectory Comparison')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
ax.legend()
ax.axis('equal')
plt.show()

# Plot 2: Position Components Comparison
fig, axs = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
pos_labels = ['X', 'Y', 'Z']
for i in range(3):
    axs[i].plot(ts_ref, r_ref[:, i], 'r--', label=f'Ground Truth {pos_labels[i]}')
    axs[i].plot(ts_jax, q_jax[:, i], 'b-', label=f'JAX {pos_labels[i]}')
    axs[i].plot(t_gazebo, q_gazebo[:, i], 'g:', label=f'Gazebo (Rosbag) {pos_labels[i]}')
    axs[i].set_ylabel(f'{pos_labels[i]} Position (m)')
    axs[i].legend()
    axs[i].grid(True)
axs[2].set_xlabel('Time (s)')
fig.suptitle('Position Tracking Comparison', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Plot 3: Attitude Comparison (Euler Angles)
fig, axs = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
angle_labels = ['Roll ($\phi$)', 'Pitch ($\theta$)', 'Yaw ($\psi$)']
for i in range(3):
    axs[i].plot(ts_jax, euler_jax[:, i], 'b-', label=f'JAX {angle_labels[i]}')
    axs[i].plot(t_gazebo, euler_gazebo[:, i], 'g:', label=f'Gazebo (Rosbag) {angle_labels[i]}')
    axs[i].set_ylabel('Angle (degrees)')
    axs[i].legend()
    axs[i].grid(True)
axs[2].set_xlabel('Time (s)')
fig.suptitle('Attitude Comparison', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()